# Agentic Orchestration: Graph & Nodes

Covers task backlog §2.1-2.6 as one file, not five. Unlike §1's notebooks (each a
data-pipeline stage handing off a file to the next), §2.1-2.5's "stages" are functions
that all have to live in one shared `StateGraph` object to actually run as a graph, so
they don't split cleanly into independent files without copy-pasting node code between
notebooks. This file grows in place as each part lands:

- **§2.1 (done):** state schema, node graph shape, the one conditional edge, all as a
  compiled and executed LangGraph skeleton with stub node logic
- **§2.2 (done):** real Router node — intent classification, constraint extraction,
  safety-flag detection, one LLM call via structured output. First node to make a real
  API call; see its cost note below for how call volume is kept low
- **§2.3 (done):** real Planner node — rule-based (no LLM call, see its section for why),
  decides `use_web` from a keyword rule; `use_rag` is always `True`
- **§2.4 (done):** real Retriever node — calls `search()` (copied from §1.4, see its
  section for why not imported) against the real index; `web.search` genuinely doesn't
  exist yet, so `use_web=True` logs a real partial-tool-failure rather than faking a result
- **§2.5 (done):** real Answerer/Critic node — one LLM call synthesizes the spoken
  recommendation + citations, but grounding and the ~15s length budget are enforced
  *deterministically* afterward, not trusted to the prompt alone
- **§2.6 (done):** real LLM provider branching (`get_chat_model()`, switches on
  `LLM_PROVIDER`, not just the model string) + a central prompt/logging wrapper every
  node call goes through. Only the `openai` path is verified — the brief requires the
  code be swappable, not every provider proven live; see its section for why

**§2.7 (testing) stays a separate file** once it exists, mirroring §1.4/§1.5: it
consumes the finished graph rather than building it.

Diagrams: [graph_state_flow.mermaid](../../Architecture/graph_state_flow.mermaid) (state
read/write per node) and
[architecture_with_mcp.mermaid](../../Architecture/architecture_with_mcp.mermaid)
(system-level view, including MCP).

**File format note:** this stays a notebook for now, per the project's file-format
convention (design/prototyping can happen in `.ipynb`). It gets exported to an
importable `.py` module once §5's Streamlit UI needs to `import` the compiled graph
at query time.

In [1]:
# !pip install langgraph langchain-openai python-dotenv chromadb sentence-transformers
# langchain-openai is used starting §2.2 (Router) -- the only node that makes a real
# LLM call so far. chromadb/sentence-transformers are used starting §2.4 (Retriever),
# reading the same persisted index §1.3 built (no new indexing here). Kept to one LLM
# call per test query throughout this notebook to control cost; see the "Cost note"
# under §2.2 below.

## Setup & config

In [2]:
import os

from dotenv import load_dotenv

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
load_dotenv(os.path.join(REPO_ROOT, ".env"))

LLM_PROVIDER = os.environ.get("LLM_PROVIDER", "openai")
LLM_MODEL = os.environ.get("LLM_MODEL", "")
_has_key = bool(os.environ.get("OPENAI_API_KEY"))  # presence only -- never read/printed

print(f"LLM_PROVIDER={LLM_PROVIDER!r}, LLM_MODEL={LLM_MODEL!r}, OPENAI_API_KEY set={_has_key}")
assert _has_key, "OPENAI_API_KEY missing -- copy .env.example to .env and fill it in"
assert LLM_MODEL, "LLM_MODEL missing in .env"

LLM_PROVIDER='openai', LLM_MODEL='gpt-5.6-luna', OPENAI_API_KEY set=True


## State schema

Matches the 8 fields from the original backlog list exactly (`query, transcript,
constraints, safety_flag, plan, tool_results, citations, final_answer`) -- kept simple,
no multi-turn/history tracking yet, revisit later if a follow-up feature needs it.
`tool_results` is one nested dict rather than three separate top-level fields.

In [3]:
from typing import Any, Dict, List, Literal, Optional, TypedDict


class Plan(TypedDict):
    """Written by the Planner node. Pure data -- no tool calls happen here."""

    use_rag: bool
    use_web: bool


class ToolResults(TypedDict):
    """Written by the Retriever node after calling MCP tools (rag.search / web.search)."""

    products: List[Dict[str, Any]]
    conflicts: List[Dict[str, Any]]
    errors: List[Dict[str, Any]]


class GraphState(TypedDict):
    """
    Shared state passed between every node in the graph. See
    ../../Architecture/graph_state_flow.mermaid for which node reads/writes what.
    """

    transcript: str  # raw ASR output (input)
    query: str  # Router: normalized search query
    constraints: Dict[str, Any]  # Router: budget/brand/category/etc.
    safety_flag: Optional[str]  # Router: reason string, or None
    plan: Plan  # Planner: sourcing decision
    tool_results: ToolResults  # Retriever: reconciled MCP results
    citations: List[Dict[str, str]]  # Answerer: doc_id/url per claim
    final_answer: Optional[str]  # Answerer: spoken-summary text


print("GraphState fields:", list(GraphState.__annotations__))

GraphState fields: ['transcript', 'query', 'constraints', 'safety_flag', 'plan', 'tool_results', 'citations', 'final_answer']


## Node graph sketch

Four real nodes plus one branch, matching `graph_state_flow.mermaid`:

`Router -> Planner -> Retriever -> Answerer/Critic`, with a `safety_flag` branch out of
Router straight to a safe-refusal path that skips Planner and Retriever entirely.

Every node below is a **stub** as of §2.1 -- sanctioned by the CP1 milestone checklist
("MCP server skeleton + tool schemas, implementation can still be stubbed"). Each gets
overwritten by a real definition in its own numbered section further down as that part
lands (Router's stub is replaced under §2.2 below, Planner's under §2.3, etc.) -- by the
time the graph actually gets built later in this notebook, it picks up whichever
definitions ran most recently, real or still-stubbed:
- Router's intent classification / constraint extraction / safety detection -> §2.2 (done)
- Planner's rag-vs-web sourcing rule -> §2.3 (done)
- Retriever's actual MCP `rag.search` / `web.search` calls -> §2.4 (done, stubbed against
  §1.4's `search()` directly until §3's MCP server exists)
- Answerer's grounding / safety / length checks -> §2.5 (done)

In [4]:
def router_node(state: GraphState) -> GraphState:
    """Stub. Real version (§2.2): LLM call classifies intent, extracts
    constraints, and sets safety_flag from the transcript. Here: passes the
    transcript through as the query untouched, no constraints extracted.
    safety_flag is left as whatever the caller passed in (used below to
    exercise the safe-refusal branch), since real detection doesn't exist yet."""
    state["query"] = state["transcript"]
    state["constraints"] = {}
    return state


def safe_refusal_node(state: GraphState) -> GraphState:
    """Reached only when Router's safety_flag is set. Skips Planner/Retriever."""
    state["final_answer"] = f"I can't help with that: {state['safety_flag']}"
    return state


def planner_node(state: GraphState) -> GraphState:
    """Stub. Real version (§2.3): LLM call decides use_web based on whether
    the query implies live price/availability. Here: rag always, web never."""
    state["plan"] = {"use_rag": True, "use_web": False}
    return state


def retriever_node(state: GraphState) -> GraphState:
    """Stub. Real version (§2.4): calls MCP rag.search, and web.search only
    if plan['use_web'], then reconciles by brand/title. Here: empty results,
    proving the node ran (the placeholder error below gets cleared)."""
    state["tool_results"] = {"products": [], "conflicts": [], "errors": []}
    return state


def answerer_node(state: GraphState) -> GraphState:
    """Stub. Real version (§2.5): LLM call synthesizes a grounded, cited,
    <=15s summary from tool_results. Here: placeholder text."""
    state["citations"] = []
    state["final_answer"] = "(stub) no answer synthesized yet"
    return state


print("5 node functions defined: router, safe_refusal, planner, retriever, answerer")

5 node functions defined: router, safe_refusal, planner, retriever, answerer


## §2.2 Router Node: real implementation

Replaces the `router_node` stub above with a real single LLM call that does intent
classification, constraint extraction, and safety-flag detection together (`with_structured_output`
against a Pydantic schema, so parsing is enforced by the API rather than by prompt-hoped-for JSON).

System prompt lives in its own versioned file, not inline in this notebook:
[Code/6. Prompts & Documentation/router_system_prompt.md](../6.%20Prompts%20%26%20Documentation/router_system_prompt.md)
(§6 Prompt Disclosure requirement — every node's prompt has to be traceable to a file, not buried in code).

**Cost note (per user request):** the LLM is used here because intent classification /
constraint extraction is the actual job it's needed for — not swapped for a rule-based
version. But call *volume* is kept deliberately low: one smoke-test call, a small
(3-query) intent/constraint test set below, plus two more calls inside the full-graph
"Test run" section further down (which now exercises the *real* safety classification
instead of a hand-set flag) — under 10 real calls total in this notebook. Router is also
the only node making real API calls so far — Planner, Retriever, Answerer stay stubs
until §2.3-2.5.

In [5]:
from pydantic import BaseModel, Field


class RouterConstraints(BaseModel):
    budget_min: Optional[float] = Field(None, description="Minimum price in USD, if stated")
    budget_max: Optional[float] = Field(None, description="Maximum price in USD, if stated")
    brand: Optional[str] = Field(None, description="Brand name, if stated")
    category: Optional[str] = Field(None, description="Product category/sub-category, if stated")
    material: Optional[str] = Field(None, description="Material, if stated")
    quantity: Optional[int] = Field(None, description="Quantity/pack size, if stated")


class RouterOutput(BaseModel):
    intent: Literal["product_search", "comparison", "follow_up", "out_of_scope"]
    query: str = Field(description="The transcript rewritten as a clean search query")
    constraints: RouterConstraints
    safety_flag: Optional[str] = Field(None, description="Reason the request itself is unsafe, else null")


ROUTER_PROMPT_PATH = os.path.join(
    REPO_ROOT, "Code", "6. Prompts & Documentation", "router_system_prompt.md"
)
with open(ROUTER_PROMPT_PATH) as f:
    ROUTER_SYSTEM_PROMPT = f.read()

print(f"Loaded router system prompt: {len(ROUTER_SYSTEM_PROMPT)} chars from {os.path.basename(ROUTER_PROMPT_PATH)}")

Loaded router system prompt: 2459 chars from router_system_prompt.md


In [6]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

router_llm = ChatOpenAI(model=LLM_MODEL, temperature=0).with_structured_output(RouterOutput)


def classify_transcript(transcript: str) -> RouterOutput:
    """The router's one LLM call, returning the full structured result
    (including `intent`, which router_node below does NOT persist to
    GraphState -- see the note above)."""
    return router_llm.invoke(
        [SystemMessage(content=ROUTER_SYSTEM_PROMPT), HumanMessage(content=transcript)]
    )


def router_node(state: GraphState) -> GraphState:
    """Real version (§2.2): classifies via classify_transcript(), then maps
    the result onto GraphState's 8 fields. Overwrites the stub above -- this
    is the definition StateGraph picks up below.

    Known gap: `intent` (product_search/comparison/follow_up/out_of_scope) is
    computed but not stored -- GraphState has no field for it (kept to the
    agreed-on 8 fields, §2.1). Practically, an out_of_scope query just flows
    through Planner/Retriever/Answerer like any other and will likely surface
    as a "no good matches" answer rather than an explicit refusal. A dedicated
    short-circuit (mirroring the safety_flag branch) is a reasonable addition
    later if that proves unsatisfying -- not adding it preemptively."""
    result = classify_transcript(state["transcript"])
    state["query"] = result.query
    state["constraints"] = result.constraints.model_dump(exclude_none=True)
    state["safety_flag"] = result.safety_flag
    return state


print("router_node redefined with real LLM call (overwrites the §2.1 stub)")

router_node redefined with real LLM call (overwrites the §2.1 stub)


In [7]:
EMPTY_STATE: GraphState = {
    "transcript": "",
    "query": "",
    "constraints": {},
    "safety_flag": None,
    "plan": {"use_rag": None, "use_web": None},
    "tool_results": {"products": [], "conflicts": [], "errors": []},
    "citations": [],
    "final_answer": None,
}

### Smoke test

One cheap real call before spending the test-set budget below, to fail fast on
auth/model-name problems rather than burning through all five test queries first.

In [8]:
_smoke = classify_transcript("insulated lunch box for kids under $20")
print("Smoke test OK.")
print(_smoke.model_dump())

Smoke test OK.
{'intent': 'product_search', 'query': 'insulated lunch box for kids', 'constraints': {'budget_min': None, 'budget_max': 20.0, 'brand': None, 'category': 'lunch box', 'material': None, 'quantity': None}, 'safety_flag': None}


### Intent/constraint test set

Three more real calls covering the remaining intents (smoke test above already covered
`product_search`). Safety-flag detection is tested separately below, inside the full
graph run, rather than a fourth call here.

In [9]:
test_transcripts = {
    "comparison": "compare the Wildkin lunch box to the Bentgo one",
    "follow_up": "what about something cheaper",
    "out_of_scope": "what's the weather like today",
}

for expected_intent, transcript in test_transcripts.items():
    result = classify_transcript(transcript)
    match = "OK" if result.intent == expected_intent else "MISMATCH"
    print(f"[{match}] expected={expected_intent!r} got={result.intent!r} | transcript={transcript!r}")
    print("   query:", result.query, "| constraints:", result.constraints.model_dump(exclude_none=True))

[OK] expected='comparison' got='comparison' | transcript='compare the Wildkin lunch box to the Bentgo one'
   query: Compare the Wildkin lunch box and the Bentgo lunch box | constraints: {'category': 'lunch box'}


[OK] expected='follow_up' got='follow_up' | transcript='what about something cheaper'
   query: something cheaper | constraints: {}


[OK] expected='out_of_scope' got='out_of_scope' | transcript="what's the weather like today"
   query: weather today | constraints: {}


## §2.3 Planner Node: real implementation

Replaces the `planner_node` stub above. Unlike Router, this is **rule-based, no LLM
call** — the decision is a single keyword-driven binary (`use_web`), and the backlog's
own framing of the rule is already keyword-based ("skip web.search unless query implies
current/latest/now"), so an extra LLM call here isn't worth the added cost/latency (per
the cost note under §2.2). `use_rag` is always `True` by design (every query gets a
catalog lookup) — see [graph_state_flow.mermaid](../../Architecture/graph_state_flow.mermaid).

Rule + rationale written up in
[Code/6. Prompts & Documentation/planner_rules.md](../6.%20Prompts%20%26%20Documentation/planner_rules.md)
(the backlog's CP2 "planner rules" deliverable) — that file also covers "fields to
retrieve / comparison criteria" as a static policy, not a new state field (the catalog's
available fields don't vary per query, so there's nothing there for the Planner to
actually decide per call).

Being rule-based, tests below are free — no reason to keep the test set small.

In [10]:
WEB_SIGNAL_KEYWORDS = [
    "current", "latest", "now", "today", "right now", "currently",
    "in stock", "available", "still available", "on sale",
]


def decide_use_web(query: str) -> bool:
    """Keyword rule from planner_rules.md. Case-insensitive substring match."""
    q = query.lower()
    return any(kw in q for kw in WEB_SIGNAL_KEYWORDS)


def planner_node(state: GraphState) -> GraphState:
    """Real version (§2.3): rule-based, no LLM call. Overwrites the stub
    above -- this is the definition StateGraph picks up below."""
    state["plan"] = {"use_rag": True, "use_web": decide_use_web(state["query"])}
    return state


print("planner_node redefined with rule-based decision (overwrites the §2.1 stub)")

planner_node redefined with rule-based decision (overwrites the §2.1 stub)


In [11]:
planner_test_cases = [
    ("insulated lunch box for kids under $20", False),
    ("what's the current price of the Wildkin lunch box", True),
    ("is this bedding set still available", True),
    ("cheap kids storage bins", False),
    ("party balloons in stock near me", True),
    ("latest toy storage organizer", True),
    ("birthday party decorations", False),
]

for query, expected in planner_test_cases:
    got = decide_use_web(query)
    match = "OK" if got == expected else "MISMATCH"
    print(f"[{match}] use_web={got!s:5} expected={expected!s:5} | {query!r}")

assert all(decide_use_web(q) == exp for q, exp in planner_test_cases), "keyword rule mismatch above"
print("\nAll", len(planner_test_cases), "cases passed (0 LLM calls -- rule-based).")

[OK] use_web=False expected=False | 'insulated lunch box for kids under $20'
[OK] use_web=True  expected=True  | "what's the current price of the Wildkin lunch box"
[OK] use_web=True  expected=True  | 'is this bedding set still available'
[OK] use_web=False expected=False | 'cheap kids storage bins'
[OK] use_web=True  expected=True  | 'party balloons in stock near me'
[OK] use_web=True  expected=True  | 'latest toy storage organizer'
[OK] use_web=False expected=False | 'birthday party decorations'

All 7 cases passed (0 LLM calls -- rule-based).


## §2.4 Retriever Node: real implementation

Replaces the `retriever_node` stub above. `search()` below is **copied from
[1.4 hybrid retrieval.ipynb](../1.%20Data%20Pipeline%20%26%20Indexing/1.4%20hybrid%20retrieval.ipynb),
not imported** — matching how [1.5 rag evaluation.ipynb](../1.%20Data%20Pipeline%20%26%20Indexing/1.5%20rag%20evaluation.ipynb)
already reused it (redefined in-notebook, reading the same persisted Chroma dirs via
`.env`; Jupyter notebooks aren't meant to import each other, and every notebook in this
project stays self-contained). Not extracted into a shared `.py` module either — that's
explicitly §3.2's job ("wraps §1.4's `search()` logic" in a module the MCP server
imports at request time), and nothing is a running process yet to justify a `.py` file
today per the file-format convention.

Real behavior, no fabricated scenarios:
- **`rag.search` always runs** (`plan.use_rag` is always `True`) — calls `search()`
  against the real `products_home_kitchen` index, no mocking
- **`web.search` doesn't exist yet** (§3.3 is decided — Tavily — but not implemented).
  When `plan.use_web` is `True`, this is logged as a genuine partial-tool-failure in
  `tool_results.errors`, and the node continues with rag results only — this is the
  actual "partial failures" backlog item, not a simulated one
- **`conflicts` stays empty for now** — reconciling `rag.search` vs `web.search` by
  brand/title needs a second result set to disagree with, which doesn't exist until
  §3.3 lands. The reconciliation logic itself gets exercised then, not faked here
- `material`/`quantity` (extracted by Router, §2.2) aren't usable filters — §1.4's
  `search()` was never built to support them (only `budget_min/max`, `category`,
  `brand`). Passed through and silently dropped for now — a real, stated gap, not
  something to quietly pretend works

**New finding while wiring this up (not previously discovered — §1.4/§1.5 never
actually exercised `search()`'s `category` parameter with a real value):**
`products_home_kitchen`'s `category_top` field is **uniformly `"Home & Kitchen"`
across all 706 rows** — it's the field that was used to *carve out* this demo slice in
§1.1/§1.2, so filtering *within* the slice by it is structurally meaningless (it can
only ever match everything or nothing). Router's free-text `category` extraction (e.g.
`"lunch box"`) was silently returning **zero results** by being passed straight through
to this dead filter — first caught by the "products returned > 0" assertion below
failing on a real query, not by manual inspection. Fixed by dropping `category` from
Retriever's hard-filter set for now (joins `material`/`quantity` as a logged, dropped
constraint) rather than quietly letting it zero out real queries. The finer-grained
sub-category breakdown (Home Décor/Furniture/Bedding/etc., §1.2) lives in
`category_full` instead, which is a full pipe-delimited hierarchy string — not a clean
exact-match target for free-text LLM extraction either; revisit if this proves worth
solving properly rather than dropping.

No LLM calls in this section (local embedding model + Chroma only, same as §1).

In [12]:
import chromadb
from sentence_transformers import SentenceTransformer

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

RETRIEVAL_EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "all-MiniLM-L6-v2")
CHROMA_PERSIST_DIR_HK = os.path.join(
    REPO_ROOT,
    os.environ.get(
        "CHROMA_PERSIST_DIR_HOME_KITCHEN",
        "Code/1. Data Pipeline & Indexing/processed/chroma_db_home_kitchen",
    ),
)
CHROMA_COLLECTION_HK = os.environ.get("CHROMA_COLLECTION", "products_home_kitchen")

retrieval_model = SentenceTransformer(RETRIEVAL_EMBEDDING_MODEL)
retrieval_client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR_HK)
retrieval_collection = retrieval_client.get_collection(CHROMA_COLLECTION_HK)

print(f"{CHROMA_COLLECTION_HK}: {retrieval_collection.count():,} products loaded for retrieval")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

products_home_kitchen: 706 products loaded for retrieval


In [13]:
def _unit_aware_tie_break(candidates, group_tolerance=0.15):
    """Copied verbatim from §1.4. Groups candidates by price_unit; within a
    group, if the group's own distance spread is tight enough to be genuine
    substitutes, re-sorts just that group by price-per-unit (cheapest first)."""
    if not candidates:
        return candidates
    result = list(candidates)

    by_unit = {}
    for i, c in enumerate(result):
        if c["price_per_unit"] is not None and c["price_unit"] is not None:
            by_unit.setdefault(c["price_unit"], []).append(i)

    for unit, idxs in by_unit.items():
        if len(idxs) < 2:
            continue
        group = [result[i] for i in idxs]
        spread = max(c["distance"] for c in group) - min(c["distance"] for c in group)
        if spread <= group_tolerance:
            for i, c in zip(idxs, sorted(group, key=lambda c: c["price_per_unit"])):
                result[i] = c
    return result


def search(query, top_k=5, budget_min=None, budget_max=None, category=None, brand=None, overfetch=30):
    """Copied verbatim from §1.4 (single-collection version -- Retriever only
    ever searches products_home_kitchen, unlike §1.4 which also demonstrated
    products_all). Hybrid retrieval: vector similarity + metadata filters,
    with unit-aware tie-breaking. Returns a list of dicts, each carrying
    `doc_id` for citation."""
    where_clauses = []
    if budget_min is not None:
        where_clauses.append({"price": {"$gte": budget_min}})
    if budget_max is not None:
        where_clauses.append({"price": {"$lte": budget_max}})
    if category is not None:
        where_clauses.append({"category_top": category})
    where = None
    if len(where_clauses) == 1:
        where = where_clauses[0]
    elif len(where_clauses) > 1:
        where = {"$and": where_clauses}

    query_embedding = retrieval_model.encode([query])[0].tolist()
    n_results = overfetch if brand else top_k
    results = retrieval_collection.query(query_embeddings=[query_embedding], n_results=n_results, where=where)

    candidates = []
    for meta, dist, doc in zip(results["metadatas"][0], results["distances"][0], results["documents"][0]):
        if brand and brand.lower() not in meta.get("brand", "").lower():
            continue
        candidates.append({
            "doc_id": meta["id"],
            "title": meta.get("title", doc),
            "price": meta.get("price"),
            "price_per_unit": meta.get("price_per_unit"),
            "price_unit": meta.get("price_unit"),
            "brand": meta.get("brand"),
            "category_top": meta.get("category_top"),
            "url": meta.get("url"),
            "distance": dist,
        })

    candidates = _unit_aware_tie_break(candidates)
    return candidates[:top_k]


print("search() and _unit_aware_tie_break() copied in from §1.4")

search() and _unit_aware_tie_break() copied in from §1.4


In [14]:
SEARCH_FILTER_KEYS = ("budget_min", "budget_max", "brand")
# "category" deliberately excluded -- see the finding in the section markdown above:
# category_top is constant across products_home_kitchen, so filtering by it here is
# structurally meaningless, not just unreliable. Joins material/quantity as a dropped,
# logged constraint rather than silently zeroing out real queries.


def retriever_node(state: GraphState) -> GraphState:
    """Real version (§2.4). Overwrites the stub above -- this is the
    definition StateGraph picks up below. See the section markdown above for
    what's real vs. genuinely-not-built-yet here."""
    products, conflicts, errors = [], [], []

    filters = {k: v for k, v in state["constraints"].items() if k in SEARCH_FILTER_KEYS}
    dropped = set(state["constraints"]) - set(filters)  # e.g. category/material/quantity
    if dropped:
        errors.append({"tool": "rag.search", "error": f"unsupported filter(s) ignored: {sorted(dropped)}"})

    try:
        rag_results = search(state["query"], **filters)
        for r in rag_results:
            r["source"] = "rag"
        products.extend(rag_results)
    except Exception as e:
        errors.append({"tool": "rag.search", "error": str(e)})

    if state["plan"]["use_web"]:
        errors.append({
            "tool": "web.search",
            "error": "not implemented yet (§3.3 decided on Tavily but not built) -- continuing with rag.search results only",
        })

    state["tool_results"] = {"products": products, "conflicts": conflicts, "errors": errors}
    return state


print("retriever_node redefined with real search() calls (overwrites the §2.1 stub)")

retriever_node redefined with real search() calls (overwrites the §2.1 stub)


### Test set (real index, no LLM calls -- free to test thoroughly)

Three cases: a normal search with a real budget filter, `use_web=True` to prove the
partial-failure logging is real (not simulated), and an unsupported constraint
(`material`) to prove the dropped-filter logging fires too.

In [15]:
# Case 1: normal rag-only search with a real budget filter
r1 = retriever_node({
    **EMPTY_STATE,
    "query": "insulated lunch box for kids",
    "constraints": {"budget_max": 20.0},
    "plan": {"use_rag": True, "use_web": False},
})["tool_results"]
print("-- Case 1: normal search --")
print(f"{len(r1['products'])} products, {len(r1['errors'])} errors, {len(r1['conflicts'])} conflicts")
for p in r1["products"][:3]:
    print(f"  [{p['distance']:.3f}] {p['title'][:60]}  (${p['price']:.2f})  doc_id={p['doc_id'][:8]}...")
assert len(r1["products"]) > 0, "expected real results for a well-covered query"
assert all(p["price"] <= 20.0 for p in r1["products"]), "budget_max filter should hold"
assert len(r1["errors"]) == 0

# Case 2: use_web=True -- web.search doesn't exist yet, should log a real partial failure
r2 = retriever_node({
    **EMPTY_STATE,
    "query": "insulated lunch box for kids",
    "constraints": {},
    "plan": {"use_rag": True, "use_web": True},
})["tool_results"]
print("\n-- Case 2: use_web=True (web.search not built yet) --")
print(f"{len(r2['products'])} products (rag still ran), errors: {r2['errors']}")
assert len(r2["products"]) > 0, "rag.search should still have run"
assert any(e["tool"] == "web.search" for e in r2["errors"]), "web.search gap should be logged"

# Case 3: unsupported constraint (material) -- should be dropped and logged, not silently ignored
r3 = retriever_node({
    **EMPTY_STATE,
    "query": "kids bedding set",
    "constraints": {"material": "cotton"},
    "plan": {"use_rag": True, "use_web": False},
})["tool_results"]
print("\n-- Case 3: unsupported constraint (material) --")
print(f"{len(r3['products'])} products, errors: {r3['errors']}")
assert any("material" in e["error"] for e in r3["errors"]), "unsupported filter should be logged"

print("\nAll 3 retriever cases passed.")

-- Case 1: normal search --
5 products, 0 errors, 0 conflicts
  [0.589] Wildkin Kids Insulated Lunch Box for Boys and Girls, Perfect  ($12.77)  doc_id=7c3f2bf5...
  [0.649] Wildkin Kids Insulated Lunch Box for Boys and Girls, Perfect  ($10.88)  doc_id=5d953e35...
  [0.687] Wildkin 33600 Monsters Lunch Box, Insulated, Moisture Resist  ($14.99)  doc_id=b1904b27...

-- Case 2: use_web=True (web.search not built yet) --
5 products (rag still ran), errors: [{'tool': 'web.search', 'error': 'not implemented yet (§3.3 decided on Tavily but not built) -- continuing with rag.search results only'}]

-- Case 3: unsupported constraint (material) --
5 products, errors: [{'tool': 'rag.search', 'error': "unsupported filter(s) ignored: ['material']"}]

All 3 retriever cases passed.


## §2.5 Answerer/Critic Node: real implementation

Replaces the `answerer_node` stub above. One structured-output LLM call synthesizes the
spoken recommendation + citations (same pattern as Router, §2.2) — but **grounding and
the length limit are enforced deterministically in Python afterward, not trusted to the
prompt alone**. A prompt instruction is a request; a citation compared against the
actual list of retrieved `doc_id`s is a fact.

System prompt: [Code/6. Prompts & Documentation/answerer_critic_system_prompt.md](../6.%20Prompts%20%26%20Documentation/answerer_critic_system_prompt.md).

- **Grounding check (real, not aspirational):** after the LLM responds, every cited
  `doc_id` is checked against `tool_results.products`. Any citation not in that set
  causes the *entire* answer to be discarded and replaced with a deterministic fallback
  — no partial credit, no attempt to salvage/edit the hallucinated text.
- **Length check:** spoken-summary budget is ~15s of speech (~40 words). Enforced by
  word count after the call, not re-prompted — a response over budget gets truncated
  deterministically. (Known rough edge: truncation can cut mid-sentence; a re-prompt
  loop would fix that but costs a second call per violation — not worth it unless this
  proves to be a frequent problem in practice.)
- **No separate chemical-safety re-check here:** Router (§2.2) already routes unsafe
  requests to `safe_refusal` before Planner/Retriever/Answerer ever run — by
  construction, Answerer never sees a flagged-unsafe query. Re-checking here would be
  redundant, not an extra safety layer.
- **Empty results skip the LLM call entirely** — if `tool_results.products` is empty,
  there is nothing an LLM call could add (the correct answer is knowable in advance:
  say nothing was found), so `answerer_node` returns a fixed fallback string directly.
  Cheaper than a call, and zero hallucination risk on a case a rule already covers
  completely — same reasoning as Planner's rule-based §2.3 design. Verified below with a
  real state (no products), not assumed.

In [16]:
class AnswererCitation(BaseModel):
    doc_id: str
    url: Optional[str] = None


class AnswererOutput(BaseModel):
    answer: str = Field(description="The spoken recommendation, ~40 words or fewer")
    citations: List[AnswererCitation] = Field(default_factory=list)


ANSWERER_PROMPT_PATH = os.path.join(
    REPO_ROOT, "Code", "6. Prompts & Documentation", "answerer_critic_system_prompt.md"
)
with open(ANSWERER_PROMPT_PATH) as f:
    ANSWERER_SYSTEM_PROMPT = f.read()

print(f"Loaded answerer system prompt: {len(ANSWERER_SYSTEM_PROMPT)} chars from {os.path.basename(ANSWERER_PROMPT_PATH)}")

Loaded answerer system prompt: 1990 chars from answerer_critic_system_prompt.md


In [17]:
ANSWER_WORD_BUDGET = 40
NO_RESULTS_FALLBACK = "I couldn't find a good match for that in the catalog."
UNGROUNDED_FALLBACK = "I found some options but couldn't confirm the details, so I don't want to guess."

answerer_llm = ChatOpenAI(model=LLM_MODEL, temperature=0).with_structured_output(AnswererOutput)


def format_candidates(products: List[Dict[str, Any]], limit: int = 5) -> str:
    if not products:
        return "(no candidate products retrieved)"
    lines = []
    for p in products[:limit]:
        price = f"${p['price']:.2f}" if p.get("price") is not None else "price unknown"
        lines.append(f"- doc_id: {p['doc_id']} | title: {p['title']} | {price} | brand: {p.get('brand') or 'unknown'}")
    return "\n".join(lines)


def grounding_check(result: AnswererOutput, valid_doc_ids: set) -> AnswererOutput:
    """Deterministic, not LLM-judged: every cited doc_id must be one Retriever
    actually returned. Any miss discards the whole answer -- no partial credit."""
    if any(c.doc_id not in valid_doc_ids for c in result.citations):
        return AnswererOutput(answer=UNGROUNDED_FALLBACK, citations=[])
    return result


def enforce_length(result: AnswererOutput) -> AnswererOutput:
    """Deterministic word-count truncation -- see the cost/quality tradeoff
    noted in the section markdown above."""
    words = result.answer.split()
    if len(words) <= ANSWER_WORD_BUDGET:
        return result
    return AnswererOutput(answer=" ".join(words[:ANSWER_WORD_BUDGET]) + "...", citations=result.citations)


def answerer_node(state: GraphState) -> GraphState:
    """Real version (§2.5). Overwrites the stub above -- this is the
    definition StateGraph picks up below."""
    products = state["tool_results"]["products"]

    if not products:
        state["final_answer"] = NO_RESULTS_FALLBACK
        state["citations"] = []
        return state

    candidate_text = format_candidates(products)
    human_message = f"Query: {state['query']}\nConstraints: {state['constraints']}\n\nCandidates:\n{candidate_text}"
    result: AnswererOutput = answerer_llm.invoke(
        [SystemMessage(content=ANSWERER_SYSTEM_PROMPT), HumanMessage(content=human_message)]
    )

    valid_doc_ids = {p["doc_id"] for p in products}
    result = grounding_check(result, valid_doc_ids)
    result = enforce_length(result)

    state["final_answer"] = result.answer
    state["citations"] = [c.model_dump(exclude_none=True) for c in result.citations]
    return state


print("answerer_node redefined with real LLM call + deterministic grounding/length checks (overwrites the §2.1 stub)")

answerer_node redefined with real LLM call + deterministic grounding/length checks (overwrites the §2.1 stub)


### Test set

All three checks below are pure Python (0 LLM calls) -- `grounding_check` and
`enforce_length` are deterministic functions, and the empty-results case short-circuits
before ever calling the LLM. The only real Answerer call in this notebook happens inside
the full-graph "Test run" further down, reusing calls already budgeted there.

In [18]:
# Case 1: grounding_check -- one valid citation passes through, one invalid citation
# discards the whole answer
valid_ids = {"doc_id_A", "doc_id_B"}

grounded = AnswererOutput(answer="The Wildkin lunch box is a good fit.", citations=[AnswererCitation(doc_id="doc_id_A")])
out1 = grounding_check(grounded, valid_ids)
assert out1.answer == grounded.answer, "valid citation should pass through unchanged"

hallucinated = AnswererOutput(answer="The Bentgo box is great.", citations=[AnswererCitation(doc_id="doc_id_NOT_RETRIEVED")])
out2 = grounding_check(hallucinated, valid_ids)
assert out2.answer == UNGROUNDED_FALLBACK and out2.citations == [], "bad citation should discard the whole answer"
print("[OK] grounding_check: valid passes through, invalid gets discarded")

# Case 2: enforce_length -- short answer unchanged, long answer truncated to budget
short = AnswererOutput(answer="Short answer here.", citations=[])
assert enforce_length(short).answer == short.answer, "under-budget answer should be unchanged"

long_answer = AnswererOutput(answer=" ".join(["word"] * 60), citations=[])
truncated = enforce_length(long_answer)
assert len(truncated.answer.rstrip(".").split()) == ANSWER_WORD_BUDGET, "should truncate to the word budget"
assert truncated.answer.endswith("..."), "truncation should be marked, not silent"
print("[OK] enforce_length: short unchanged, long truncated to budget with a visible marker")

# Case 3: empty results -- short-circuits before any LLM call
r_empty = answerer_node({**EMPTY_STATE, "query": "something obscure", "tool_results": {"products": [], "conflicts": [], "errors": []}})
assert r_empty["final_answer"] == NO_RESULTS_FALLBACK
assert r_empty["citations"] == []
print("[OK] empty results: deterministic fallback, no LLM call, no invented product")

print("\nAll §2.5 unit tests passed (0 LLM calls).")

[OK] grounding_check: valid passes through, invalid gets discarded
[OK] enforce_length: short unchanged, long truncated to budget with a visible marker
[OK] empty results: deterministic fallback, no LLM call, no invented product

All §2.5 unit tests passed (0 LLM calls).


## §2.6 LLM Provider Abstraction

Two things, both from the assignment brief's actual **Model-Agnostic Requirement**:
*"Teams may use any LLM (commercial or open) as long as the code is swappable via
environment variables/config"* and *"Log prompts & system messages for transparency"*
(a separate, named grading line: Prompt Disclosure). Not project-invented scope --
and, checked against the brief's exact wording, not more than that either: it does
**not** say every listed provider has to actually be tested, live or mocked. That
stricter bar ("confirm tool/function calling works against at least 2 providers") was
this project's own backlog adding rigor beyond the brief -- dropped here per user
request, since it doesn't serve the actual requirement.

**Provider branching**, not just model-string config: `get_chat_model()` below actually
switches on `LLM_PROVIDER` (openai/anthropic/google/bedrock, the brief's own listed
hosted options) instead of hardcoding `ChatOpenAI` the way §2.2's Router originally
did -- this is what makes the *code* swappable via config, which is the actual
requirement. **Only the `openai` path is verified** (it's the one actually in use);
`anthropic`/`google`/`bedrock` are written to the same interface but genuinely
untested -- not mock-verified either, an honest "unverified" rather than a disguised
one. No new dependencies installed for any of this.

**Central prompt/logging wrapper**: `invoke_and_log()` is now the only way any node
calls an LLM — Router (§2.2) and Answerer (§2.5) get refactored below to go through it
instead of calling `.invoke()` directly. Every call's system prompt, input, and output
lands in `LLM_CALL_LOG`, satisfying the brief's transparency requirement structurally
(a real log exists, capturing real calls) rather than just asserting it in prose.

In [19]:
def get_chat_model(provider: str, model: str, **kwargs):
    """Real provider branching (Model-Agnostic Requirement: swappable code,
    not just a config string). Each branch imports its package lazily -- only
    the actually-selected provider's package needs to be installed."""
    provider = (provider or "").lower()
    if provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=model, **kwargs)
    if provider == "anthropic":
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=model, **kwargs)
    if provider == "google":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=model, **kwargs)
    if provider == "bedrock":
        from langchain_aws import ChatBedrock
        return ChatBedrock(model_id=model, **kwargs)
    raise ValueError(f"Unknown LLM_PROVIDER: {provider!r} (expected one of: openai, anthropic, google, bedrock)")


print("get_chat_model() defined -- real branching on LLM_PROVIDER")

get_chat_model() defined -- real branching on LLM_PROVIDER


In [20]:
# Only the openai path is exercised -- the charter requires the *code* to be
# swappable via config, not that every listed provider actually be tested (that was
# this project's own backlog adding a stricter bar than the brief actually sets; see
# the section markdown above). anthropic/google/bedrock branches above are written to
# the same interface but genuinely untested, not mock-verified either -- an honest
# "unverified," not a disguised one.
from langchain_openai import ChatOpenAI as RealChatOpenAI

real_model = get_chat_model("openai", LLM_MODEL, temperature=0)
assert isinstance(real_model, RealChatOpenAI)
print("[OK] provider='openai' -> real ChatOpenAI instantiated")

# unknown provider -> a clear error, not a silent fallback to some default
try:
    get_chat_model("made_up_provider", "x")
    raise AssertionError("should have raised ValueError")
except ValueError as e:
    print(f"[OK] unknown provider raises: {e}")

[OK] provider='openai' -> real ChatOpenAI instantiated
[OK] unknown provider raises: Unknown LLM_PROVIDER: 'made_up_provider' (expected one of: openai, anthropic, google, bedrock)


### Central prompt/logging wrapper

Re-points Router (§2.2) and Answerer (§2.5) at `get_chat_model()` instead of a
hardcoded `ChatOpenAI(...)`, and routes every call through `invoke_and_log()` instead
of calling `.invoke()` directly. Behavior is unchanged (still real OpenAI calls, same
prompts) -- this only changes *how* the call is made and *that* it gets logged.

In [21]:
LLM_CALL_LOG: List[Dict[str, Any]] = []


def invoke_and_log(llm, node_name: str, system_prompt: str, human_content: str):
    """The only path any node uses to call an LLM from here on. Captures the
    system prompt + input + output of every call -- the assignment brief's
    'log prompts & system messages for transparency' (Prompt Disclosure)."""
    result = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=human_content)])
    LLM_CALL_LOG.append({
        "node": node_name,
        "system_prompt_chars": len(system_prompt),
        "input": human_content,
        "output": result.model_dump() if hasattr(result, "model_dump") else str(result),
    })
    return result


# Router (§2.2): re-point at get_chat_model() and route through invoke_and_log()
router_llm = get_chat_model(LLM_PROVIDER, LLM_MODEL, temperature=0).with_structured_output(RouterOutput)


def classify_transcript(transcript: str) -> RouterOutput:
    return invoke_and_log(router_llm, "router", ROUTER_SYSTEM_PROMPT, transcript)


# Answerer (§2.5): same treatment
answerer_llm = get_chat_model(LLM_PROVIDER, LLM_MODEL, temperature=0).with_structured_output(AnswererOutput)


def answerer_node(state: GraphState) -> GraphState:
    """Real version (§2.5), re-defined here only to route its LLM call
    through invoke_and_log() -- logic is otherwise identical to the §2.5
    definition above."""
    products = state["tool_results"]["products"]

    if not products:
        state["final_answer"] = NO_RESULTS_FALLBACK
        state["citations"] = []
        return state

    candidate_text = format_candidates(products)
    human_message = f"Query: {state['query']}\nConstraints: {state['constraints']}\n\nCandidates:\n{candidate_text}"
    result: AnswererOutput = invoke_and_log(answerer_llm, "answerer", ANSWERER_SYSTEM_PROMPT, human_message)

    valid_doc_ids = {p["doc_id"] for p in products}
    result = grounding_check(result, valid_doc_ids)
    result = enforce_length(result)

    state["final_answer"] = result.answer
    state["citations"] = [c.model_dump(exclude_none=True) for c in result.citations]
    return state


print("router_llm/answerer_llm rebuilt via get_chat_model(); classify_transcript/answerer_node now log every call")

router_llm/answerer_llm rebuilt via get_chat_model(); classify_transcript/answerer_node now log every call


## Conditional edges

Only one graph-level branch: **Router -> safe_refusal vs. Router -> Planner**, on
whether `safety_flag` got set.

The backlog's other example ("skip web.search unless query implies current/latest/now")
is *not* a second graph edge. It's a decision Planner writes into `plan['use_web']`, and
Retriever reads as a plain `if` before deciding which MCP tools to call. A graph-level
conditional edge is for skipping whole *nodes* (Planner and Retriever both get skipped on
the safety path) -- rag-vs-web is a choice inside one node, not a choice between nodes.

In [22]:
from langgraph.graph import END, StateGraph


def route_after_router(state: GraphState) -> Literal["safe_refusal", "planner"]:
    return "safe_refusal" if state.get("safety_flag") else "planner"


builder = StateGraph(GraphState)
builder.add_node("router", router_node)
builder.add_node("safe_refusal", safe_refusal_node)
builder.add_node("planner", planner_node)
builder.add_node("retriever", retriever_node)
builder.add_node("answerer", answerer_node)

builder.set_entry_point("router")
builder.add_conditional_edges(
    "router", route_after_router, {"safe_refusal": "safe_refusal", "planner": "planner"}
)
builder.add_edge("planner", "retriever")
builder.add_edge("retriever", "answerer")
builder.add_edge("answerer", END)
builder.add_edge("safe_refusal", END)

graph = builder.compile()
print("Graph compiled. Nodes:", list(graph.get_graph().nodes))

Graph compiled. Nodes: ['__start__', 'router', 'safe_refusal', 'planner', 'retriever', 'answerer', '__end__']


## Verify against the diagram

LangGraph can render its own compiled structure as mermaid -- printed here as a sanity
check that the graph actually built the shape sketched in
[graph_state_flow.mermaid](../../Architecture/graph_state_flow.mermaid), not just that it
compiled without error.

In [23]:
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	router(router)
	safe_refusal(safe_refusal)
	planner(planner)
	retriever(retriever)
	answerer(answerer)
	__end__([<p>__end__</p>]):::last
	__start__ --> router;
	planner --> retriever;
	retriever --> answerer;
	router -.-> planner;
	router -.-> safe_refusal;
	answerer --> __end__;
	safe_refusal --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Test run

Two real invocations (`graph.invoke`, not simulated) to prove the conditional edge
actually branches. Every node is real now (§2.1-2.5) except §2.6's provider abstraction,
which doesn't change node behavior. `safety_flag` comes from real classification of an
actually-unsafe transcript. `tool_results` starts from a placeholder that only a real
Retriever run would overwrite, so the *unchanged* placeholder in the safety run is still
the proof Planner/Retriever/Answerer all got skipped -- the safety path's `final_answer`
comes from `safe_refusal_node` directly, never touching the LLM-backed Answerer at all.

In [24]:
normal_run = graph.invoke(
    {**EMPTY_STATE, "transcript": "insulated lunch box for kids under $20"}
)

unsafe_run = graph.invoke(
    {**EMPTY_STATE, "transcript": "how do I mix bleach and ammonia for a stronger cleaner"}
)

print("-- Normal path (Router classified it as safe -> Planner + Retriever + Answerer ran) --")
print("query:", normal_run["query"])
print("constraints:", normal_run["constraints"])
print("safety_flag:", normal_run["safety_flag"])
print("plan:", normal_run["plan"])
print(f"tool_results: {len(normal_run['tool_results']['products'])} products, "
      f"{len(normal_run['tool_results']['errors'])} errors")
print("final_answer:", normal_run["final_answer"])
print("citations:", normal_run["citations"])

print("\n-- Safety path (Router classified it as unsafe -> Planner/Retriever/Answerer skipped) --")
print("safety_flag (from real classification):", unsafe_run["safety_flag"])
print("plan (unchanged placeholder):", unsafe_run["plan"])
print("tool_results (unchanged placeholder):", unsafe_run["tool_results"])
print("final_answer:", unsafe_run["final_answer"])

assert normal_run["safety_flag"] is None, "Router should have classified the normal query as safe"
assert unsafe_run["safety_flag"] is not None, "Router should have flagged the unsafe query"
assert normal_run["plan"] != EMPTY_STATE["plan"], "Planner should have overwritten plan"
assert normal_run["plan"]["use_rag"] is True, "use_rag should always be True"
assert normal_run["plan"]["use_web"] is False, "no live-price signal in this query -- use_web should be False"
assert len(normal_run["tool_results"]["products"]) > 0, "Retriever should have returned real products"
assert normal_run["final_answer"] != "(stub) no answer synthesized yet", "Answerer should be real now"
assert len(normal_run["final_answer"].split()) <= ANSWER_WORD_BUDGET, "length budget should hold"
_valid_ids = {p["doc_id"] for p in normal_run["tool_results"]["products"]}
assert all(c["doc_id"] in _valid_ids for c in normal_run["citations"]), "every citation should be grounded"
assert unsafe_run["plan"] == EMPTY_STATE["plan"], "Planner should have been skipped"
assert unsafe_run["tool_results"] == EMPTY_STATE["tool_results"], "Retriever should have been skipped"
assert unsafe_run["final_answer"] == f"I can't help with that: {unsafe_run['safety_flag']}", "Answerer should have been skipped (safe_refusal wrote this directly)"
print("\nAll assertions passed: real Router + Planner + Retriever + Answerer drive the graph correctly.")

-- Normal path (Router classified it as safe -> Planner + Retriever + Answerer ran) --
query: insulated lunch box for kids
constraints: {'budget_max': 20.0, 'category': 'lunch box'}
safety_flag: None
plan: {'use_rag': True, 'use_web': False}
tool_results: 5 products, 1 errors
final_answer: For a kids’ insulated lunch box under $20, consider the Wildkin Kids lunch box in Wild Animals for $10.88, or the Skip Hop Zoo insulated lunch box for $14.99.
citations: [{'doc_id': '5d953e357d08badf591931b299d2065c'}, {'doc_id': 'fbf2a19ecd880f3c1d6ebb49d3f677ad'}]

-- Safety path (Router classified it as unsafe -> Planner/Retriever/Answerer skipped) --
safety_flag (from real classification): Mixing bleach and ammonia creates toxic chloramine gas and is dangerous.
plan (unchanged placeholder): {'use_rag': None, 'use_web': None}
tool_results (unchanged placeholder): {'products': [], 'conflicts': [], 'errors': []}
final_answer: I can't help with that: Mixing bleach and ammonia creates toxic chloramine

### §2.6 continued: what the logging wrapper captured

`LLM_CALL_LOG` only starts capturing from the point §2.6 redefined `classify_transcript`/
`answerer_node` onward -- so it holds exactly the calls made by the "Test run" above:
Router runs on both invocations (it's first in the graph), Answerer only on the one that
didn't get routed to `safe_refusal`.

In [25]:
print(f"{len(LLM_CALL_LOG)} calls logged:\n")
for i, entry in enumerate(LLM_CALL_LOG):
    print(f"[{i}] node={entry['node']!r}  system_prompt_chars={entry['system_prompt_chars']}")
    print(f"    input:  {entry['input'][:80]!r}")
    print(f"    output: {str(entry['output'])[:120]!r}")

assert len(LLM_CALL_LOG) > 0, "wrapper should have captured real calls from the Test run above"
assert {e["node"] for e in LLM_CALL_LOG} == {"router", "answerer"}, "should see both node types"
print("\nWrapper is capturing real calls from both nodes -- Prompt Disclosure requirement satisfied structurally.")

3 calls logged:

[0] node='router'  system_prompt_chars=2459
    input:  'insulated lunch box for kids under $20'
    output: "{'intent': 'product_search', 'query': 'insulated lunch box for kids', 'constraints': {'budget_min': None, 'budget_max': "
[1] node='answerer'  system_prompt_chars=1990
    input:  "Query: insulated lunch box for kids\nConstraints: {'budget_max': 20.0, 'category'"
    output: "{'answer': 'For a kids’ insulated lunch box under $20, consider the Wildkin Kids lunch box in Wild Animals for $10.88, o"
[2] node='router'  system_prompt_chars=2459
    input:  'how do I mix bleach and ammonia for a stronger cleaner'
    output: "{'intent': 'out_of_scope', 'query': 'mix bleach and ammonia for a stronger cleaner', 'constraints': {'budget_min': None,"

Wrapper is capturing real calls from both nodes -- Prompt Disclosure requirement satisfied structurally.


## Next steps

- §2.7: now that all four nodes are real (including provider abstraction + logging),
  testing gets its own notebook (e.g. `2.7 testing.ipynb`) that imports/exercises this
  file's finished graph -- unit tests per node, an end-to-end test, and a regression
  set of demo queries
- §3: build the real MCP server; swap Retriever's direct `search()` call over to a real
  `rag.search` MCP tool call, and implement `web.search` for real (Tavily, §3.3) so
  `use_web=True` stops being a logged gap and `tool_results.conflicts` gets exercised
  for real
- If this project ever actually needs a non-OpenAI provider, §2.6's mock-verified
  branching should get a real live test against that provider at that point
- Once §5's Streamlit UI needs to call this graph per-query, export it to an importable
  `.py` module (reason (c) in the file-format convention) rather than re-executing this
  notebook live